In [13]:
import pandas as pd
import numpy as np
# 就是你刚才那一串sklearn导入！
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 读取数据
df = pd.read_csv("./train.csv")
print(df)

     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 ...     ...   ... 

In [14]:
# 基础探查
# 1. 查看前5行
print(df.head())

# 2. 行列形状
print(df.shape) # (891,12)

# 3. 查看数据类型、缺失值（重点！）
print(df.info())

# 4. 统计数值列：均值、最大最小、标准差（Age、Fare量纲差距巨大，正好对应你笔记的例子）
print(df.describe())

# 5. 统计每一列缺失值数量【面试高频】
print(df.isna().sum())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
(8

In [15]:
# 筛选、条件查询(Pandas取数据)
# 取单列：所有人年龄
df["Age"]
# 取多列
df[["Sex","Age","Fare","Survived"]]

# loc：按标签筛选：所有女性乘客
df.loc[df["Sex"]=="female"]

# 多条件筛选：30岁以上、一等舱、存活的人
df.loc[(df["Age"]>30) & (df["Pclass"]==1) & (df["Survived"]==1)]

# iloc：按位置下标，前10行，0~4列
df.iloc[0:10, 0:5]


,PassengerId,Survived,Pclass,Name,Sex
0,1,0,3,"Braund, Mr. Owen Harris",male
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female
2,3,1,3,"Heikkinen, Miss. Laina",female
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female
4,5,0,3,"Allen, Mr. William Henry",male
5,6,0,3,"Moran, Mr. James",male
6,7,0,1,"McCarthy, Mr. Timothy J",male
7,8,0,3,"Palsson, Master. Gosta Leonard",male
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female


In [16]:
# 新增、删除列
# 删除无用列：PassengerId, Name, Ticket, Cabin
df = df.drop(["PassengerId","Name","Ticket","Cabin"], axis=1)

# 新增特征：家庭成员总数 = SibSp + Parch
df["FamilySize"] = df["SibSp"] + df["Parch"]


In [17]:
# 缺失值处理（pandas 原生填充，和 SimpleImputer 对照理解）
# 方式1：pandas手动填充（探索阶段用）
df["Age"] = df["Age"].fillna(df["Age"].median()) # 年龄用中位数填充
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0]) # 分类用众数


In [18]:
# 分组统计 groupby（看业务规律）
# 按性别分组，看存活率
print(df.groupby("Sex")["Survived"].mean())
# 按舱位等级分组，看存活率
print(df.groupby("Pclass")["Survived"].mean())


Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64


In [19]:
# 划分 X 特征、y 标签，训练集测试集拆分
# Survived是标签，剩下全部是特征
X = df.drop("Survived", axis=1)
y = df["Survived"]

# 划分训练集、测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [20]:
# 重点：Pipeline + ColumnTransformer 流水线
# 定义数值特征流水线：填充缺失 + 标准化
num_features = ["Age", "Fare", "SibSp", "Parch", "FamilySize"]
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("std_scaler", StandardScaler()) # 标准化！就是你图片里的内容
])

# 分类特征流水线：众数填充 + 独热编码
cat_features = ["Sex", "Embarked", "Pclass"]
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 合并两条流水线，按列分配
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])


In [21]:
# 预处理+模型打包
from sklearn.linear_model import LogisticRegression

# 完整流水线：预处理 + 模型
full_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression())
])

# 训练，只fit训练集！
full_pipeline.fit(X_train, y_train)

# 测试集预测
acc = full_pipeline.score(X_test, y_test)
print(f"测试集准确率：{acc:.2f}")


测试集准确率：0.80
